In [1]:
# goal: enumerate genus-two curves with c_1 (= b_1) = 0

def test(p, modulus=0, verbose=False):
    F = GF(p)
    R.<x> = F[]
    
    # t_1 = x^p - x
    t_1 = R([0,-1]+[0]*(p-2)+[1])
    assert t_1.degree() == p
    
    # t_2 = x^{p^2} - x
    t_2 = R([0,-1]+[0]*(p^2-2)+[1])
    assert t_2.degree() == p^2
    
    # implement chi_{1,!} : F_p -> {0,1,-1}
    def sign(b):
        if not b:
            return 0
        if b == 1:
            return 1
        return -1

    # precompute all values of chi_{1,!}
    legendre = [ sign(F(a)^((p-1)//2)) for a in range(p) ]
    #print(legendre)

    # input: a (= sum_{i=0}^3 a_i*p^i in base-p)
    # output: (a0,a1,a2,a3)
    def coeffs(a):
        c = []
        for i in range(4):
            c.append(F(a))
            a //= p
        return c
    
    def symbol_to_str(symbol):
        if symbol > 0:
            return '+'
        if symbol < 0:
            return '-'
        return '0'

    stub = { True: [0,1], False: [0,0,1] }
    
    # enumerate polynomials f(x) = x^5 + sum_{i=0}^3 a_i*x^i
    p2 = p*p
    for abcd in range(p^4):
        # skip anything which is divisible by x^2
        if abcd % p2 == 0:
            continue
            
        for odd_case in [True,False]:
            # create polynomial
            c = coeffs(abcd) + stub[odd_case]
            f = R(c)

            # calculate derivative
            d = [ ai*ci for ai,ci in zip([1,2,3,4,5,6],c[1:])]
            df = R(d)
        
            # filter f which are not square free
            g = f.gcd(df)
            if g.degree() > 0:
                continue
            
            # in odd case, filter f which have an even number of zeros
            g = f.gcd(t_1)
            if odd_case and g.degree() % 2 == 0:
                continue

            # in even case, filter f which have any zeros (they're redundant)
            if not odd_case and g.degree() > 1:
                continue

            # count points
            symbols = [ legendre[f(a)] for a in range(p) ]
            c_1 = sum(symbols)

            if (modulus == c_1 == 0): # or (modulus != 0 == c_1 % modulus):
                if verbose:
                    c = [ int(ci) for ci in c ]
                    symbols = ''.join([ symbol_to_str(symbol) for symbol in symbols ])
                    print(f'{p:3} : {odd_case:1} : {c[3]:2} {c[2]:2} {c[1]:2} {c[0]:2} : {symbols}')
                yield (odd_case,abcd)
                #print(f'{abcd:4} : {f}')


In [2]:
# version two

def test(p, modulus=0, verbose=False):
    # skip p=3 and p=5 since orbits of polynomials are more complicated
    assert p > 5

    F = GF(p)
    
    R.<x> = F[]
    
    #t_1 = x^p - x
    
    # create minimal set of leading polynomials
    g = F.multiplicative_generator()
    stubs = []
    for gap in [2,3,4,5,6]:
        # find orbits of F_p^* acting by F_p^* by a.b = a^gap * b
        d = gcd(gap,p-1)

        # kernel of action is mu_d
        
        # find coset representatives of F_p^*/F_p^{*d}
        r = 1
        for _ in range(d):
            # leading stub is (x^gap + r)*x^i for some i
            c = [r] + [0]*(gap-1) + [1]
            
            # deg(stub)=6 => i = 6-gap
            stubs.append((c,6-gap))

            # deg(stub)=5 => i = 5-gap and gap < 6
            if gap < 6:
                stubs.append((c,5-gap))
            
            # prepare next representative
            r *= g

    # precompute (small) powers of p
    p_to_i = [ 1 ]
    while len(p_to_i) < 5:
        p_to_i.append(p_to_i[-1]*p)
    
    # implement chi_{1,!} : F_p -> {0,1,-1}
    def sign(b):
        if not b:
            return 0
        if b == 1:
            return 1
        return -1

    # precompute all values of chi_{1,!}
    legendre = [ sign(F(a)^((p-1)//2)) for a in range(p) ]
    #print(legendre)
    
    def symbol_to_str(symbol):
        if symbol > 0:
            return '+'
        if symbol < 0:
            return '-'
        return '0'
        
    def exp_mod(x, e, f):
        if not e:
            return 1
        y = exp_mod(x, e//2, f)
        y = (y*y) % f
        if e % 2:
            y = (y*x) % f
        return y
        
    for stub,tail_len in stubs:
        for tail in range(p_to_i[tail_len]):
            if tail % p_to_i[2] == 0:
                continue

            # extract first tail_len digits in base-p expansion of tail
            c = []
            for i in range(tail_len):
                c.append(tail % p)
                tail //= p
            c += stub
            f = R(c)

            # calculate derivative
            d = [ ai*ci for ai,ci in zip([1,2,3,4,5,6],c[1:])]
            df = R(d)
        
            # filter f which are not square free
            g = f.gcd(df)
            if g.degree() > 0:
                continue
            
            odd_case = f.degree() == 5
            even_case = not odd_case

            # in odd case, filter f which have an even number of zeros
            g = f.gcd(exp_mod(x, p, f) - x)
            if odd_case and g.degree() % 2 == 0:
                continue

            # in even case, filter f which have any zeros (they're redundant)
            if even_case and g.degree() > 1:
                continue

            # count points
            symbols = [ legendre[f(a)] for a in range(p) ]
            c_1 = sum(symbols)
            
            # Legendre symbol at infinity:
            # - 0 in odd case
            # - 1 in even case (since f is monic)
            if not odd_case:
                c_1 += 1
            
            if (modulus == c_1 == 0): # or (modulus != 0 == c_1 % modulus):
                if verbose:
                    c = [ int(ci) for ci in c ]
                    c.reverse()
                    
                    coeffs = ''.join([ f'{ci:2}' for ci in c ])
                    symbols = ''.join([ symbol_to_str(symbol) for symbol in symbols ])
                    
                    print(f'{p:3} : {odd_case:1} : {coeffs} : {symbols}')
                yield (odd_case,stub,tail)
                #print(f'{abcd:4} : {f}')
            

In [3]:
# spit out data for checking by hand
_ = list(test(11,verbose=True))

 11 : 0 :  1 0 1 0 0 0 5 : +--++--++--
 11 : 0 :  1 0 1 0 0 1 5 : +--++-+-+--
 11 : 0 :  1 0 1 0 0 1 9 : +++-+--+---
 11 : 0 :  1 0 1 0 0 2 3 : +--+-+--+-+
 11 : 0 :  1 0 1 0 0 2 8 : -++-+-+-+--
 11 : 0 :  1 0 1 0 0 3 9 : ++-+-+-+---
 11 : 0 :  1 0 1 0 0 310 : -+-+--++--+
 11 : 0 :  1 0 1 0 0 8 9 : +---+-+-+-+
 11 : 0 :  1 0 1 0 0 810 : -+--++--+-+
 11 : 0 :  1 0 1 0 0 9 3 : ++-+--+-+--
 11 : 0 :  1 0 1 0 0 9 8 : ---+-+-+-++
 11 : 0 :  1 0 1 0 010 5 : +--+-+-++--
 11 : 0 :  1 0 1 0 010 9 : +---+--+-++
 11 : 0 :  1 0 1 0 1 0 3 : +---++++---
 11 : 0 :  1 0 1 0 1 2 3 : +-+++-----+
 11 : 0 :  1 0 1 0 1 4 6 : ---++-+--++
 11 : 0 :  1 0 1 0 1 4 9 : ++-++--+---
 11 : 0 :  1 0 1 0 1 5 4 : ++--+---++-
 11 : 0 :  1 0 1 0 1 5 7 : -+-+-+---++
 11 : 0 :  1 0 1 0 1 6 4 : +-++---+--+
 11 : 0 :  1 0 1 0 1 6 7 : -++---+-+-+
 11 : 0 :  1 0 1 0 1 7 6 : -++--+-++--
 11 : 0 :  1 0 1 0 1 7 9 : +---+--++-+
 11 : 0 :  1 0 1 0 1 9 3 : ++-----+++-
 11 : 0 :  1 0 1 0 2 3 2 : -+-+++----+
 11 : 0 :  1 0 1 0 2 3 3 

 11 : 0 :  1 0 1 6 6 1 6 : ---+-++-++-
 11 : 0 :  1 0 1 6 6 3 3 : ++-+----++-
 11 : 0 :  1 0 1 6 6 3 4 : +--+-+--+-+
 11 : 0 :  1 0 1 6 6 3 9 : ++---+--++-
 11 : 0 :  1 0 1 6 6 410 : --++-++--+-
 11 : 0 :  1 0 1 6 6 6 7 : -+--++--+-+
 11 : 0 :  1 0 1 6 6 7 4 : +++----++--
 11 : 0 :  1 0 1 6 6 8 1 : +++-++-----
 11 : 0 :  1 0 1 6 6 8 2 : --+-+-+-++-
 11 : 0 :  1 0 1 6 610 7 : -++--+++---
 11 : 0 :  1 0 1 6 7 0 2 : --++---++-+
 11 : 0 :  1 0 1 6 7 5 7 : -+-+-++---+
 11 : 0 :  1 0 1 6 7 6 2 : -++-+++----
 11 : 0 :  1 0 1 6 7 6 5 : ++--+---++-
 11 : 0 :  1 0 1 6 71010 : ----+++-+-+
 11 : 0 :  1 0 1 6 8 1 6 : -++-+-+---+
 11 : 0 :  1 0 1 6 8 1 7 : --++--+-++-
 11 : 0 :  1 0 1 6 8 4 3 : ++-+-----++
 11 : 0 :  1 0 1 6 8 5 3 : +---+-+++--
 11 : 0 :  1 0 1 6 8 6 6 : ---+--+-+++
 11 : 0 :  1 0 1 6 8 610 : ---+++++---
 11 : 0 :  1 0 1 6 8 7 4 : ++--+-+---+
 11 : 0 :  1 0 1 6 8 8 3 : +++++------
 11 : 0 :  1 0 1 6 8 8 5 : +-+---+--++
 11 : 0 :  1 0 1 6 8 9 2 : -++---++-+-
 11 : 0 :  1 0 1 6 8 9 4 

 11 : 0 :  1 0 2 1 3 2 5 : +++-+-+----
 11 : 0 :  1 0 2 1 3 3 8 : --++-+-+-+-
 11 : 0 :  1 0 2 1 3 5 3 : ++---+---++
 11 : 0 :  1 0 2 1 3 6 6 : ---++-++--+
 11 : 0 :  1 0 2 1 3 8 6 : -----++++-+
 11 : 0 :  1 0 2 1 3 8 8 : -+-+---+-++
 11 : 0 :  1 0 2 1 3 8 9 : +-+-+--+-+-
 11 : 0 :  1 0 2 1 3 9 8 : ---+++--+-+
 11 : 0 :  1 0 2 1 4 0 7 : -+-+--+-++-
 11 : 0 :  1 0 2 1 4 0 8 : -+-+++----+
 11 : 0 :  1 0 2 1 4 0 9 : +---++---++
 11 : 0 :  1 0 2 1 4 2 4 : ++---++--+-
 11 : 0 :  1 0 2 1 4 3 3 : ++-+---++--
 11 : 0 :  1 0 2 1 4 310 : --++-++--+-
 11 : 0 :  1 0 2 1 4 4 8 : -++--+--++-
 11 : 0 :  1 0 2 1 4 5 4 : +--+---++-+
 11 : 0 :  1 0 2 1 4 6 5 : +-++-----++
 11 : 0 :  1 0 2 1 4 9 1 : +--+--+-+-+
 11 : 0 :  1 0 2 1 410 1 : +-+-+-++---
 11 : 0 :  1 0 2 1 5 0 6 : -+++-+---+-
 11 : 0 :  1 0 2 1 5 110 : -++-+---+-+
 11 : 0 :  1 0 2 1 5 2 1 : ++--+++----
 11 : 0 :  1 0 2 1 5 2 2 : --++--+++--
 11 : 0 :  1 0 2 1 5 3 5 : +-++----+-+
 11 : 0 :  1 0 2 1 5 4 3 : +++-----++-
 11 : 0 :  1 0 2 1 510 9 

 11 : 0 :  1 0 2 8 0 8 1 : +++-+---+--
 11 : 0 :  1 0 2 8 0 810 : ----+-++++-
 11 : 0 :  1 0 2 8 0 9 6 : -+--+---+++
 11 : 0 :  1 0 2 8 010 9 : +----++-+-+
 11 : 0 :  1 0 2 8 1 0 5 : +-+-+--+--+
 11 : 0 :  1 0 2 8 1 0 7 : ---+--+++-+
 11 : 0 :  1 0 2 8 1 1 2 : -++-+---++-
 11 : 0 :  1 0 2 8 1 2 3 : +--+-+++---
 11 : 0 :  1 0 2 8 1 3 9 : +-++++-----
 11 : 0 :  1 0 2 8 1 4 2 : --+--++-+-+
 11 : 0 :  1 0 2 8 1 4 3 : +--+-+--++-
 11 : 0 :  1 0 2 8 1 510 : -+---++--++
 11 : 0 :  1 0 2 8 1 6 1 : +-+--++--+-
 11 : 0 :  1 0 2 8 2 0 2 : -+++---+-+-
 11 : 0 :  1 0 2 8 2 1 7 : --+-+---+++
 11 : 0 :  1 0 2 8 2 4 3 : +++-+--+---
 11 : 0 :  1 0 2 8 2 4 4 : +-+---+-++-
 11 : 0 :  1 0 2 8 2 5 1 : +-+-+----++
 11 : 0 :  1 0 2 8 2 5 5 : ++-+-+--+--
 11 : 0 :  1 0 2 8 2 6 4 : ++-+--++---
 11 : 0 :  1 0 2 8 2 6 6 : -+-+++---+-
 11 : 0 :  1 0 2 8 2 8 5 : ++-----++-+
 11 : 0 :  1 0 2 8 2 9 2 : --+-+++---+
 11 : 0 :  1 0 2 8 2 9 8 : ---++++-+--
 11 : 0 :  1 0 2 8 210 1 : +--++-+-+--
 11 : 0 :  1 0 2 8 3 0 1 

In [4]:
# WARNING: overwrites d
#assert False
d = {}

In [5]:
# goal: unpickle previously computed d for use as d0
import pickle

with open("genus-2--primes-3-97.pkl", "rb") as f:
    d0 = pickle.load(f)

count = 0
for k,v in d0.items():
    print(k,len(v))
    count += len(v)
print(f'total : {count}')

7 644
11 2838
13 5538
17 13432
19 20782
23 39109
29 90030
31 116754
37 219617
41 306810
43 370541
47 497277
53 769024
59 1121942
61 1276154
67 1777350
71 2162226
73 2420584
79 3189346
83 3767757
89 4828979
97 6591275
total : 29588009


In [6]:
from datetime import datetime
print(datetime.now())

2026-05-23 06:41:20.532083


In [7]:
# WARNING: very slow code; will overwrite d

start = middle = datetime.now()

count = 0
print("  p |      p^4 |     p^3 |   delta | expected |    total |     time | log_p(delta)")
print("----|----------|---------|---------|----------|----------|----------|--------------")
for p in primes(7,20):
    d[p] = list(test(p))
    delta = len(d[p])
    count += delta
    s = str(datetime.now()-middle).split('.')[0]
    if len(s) < 8:
        s = ' ' + s
    print(f'{p:3} | {p^4:8} : {p^3:7} | {delta:7} : {len(d0[p]):8} | {count:8} | {s:8} | {round(log(delta)/log(p),3)}')
    middle = datetime.now()

print()
print(f"total time: {datetime.now()-start}")

  p |      p^4 |     p^3 |   delta | expected |    total |     time | log_p(delta)
----|----------|---------|---------|----------|----------|----------|--------------
  7 |     2401 :     343 |     641 :      644 |      641 |  0:00:00 | 3.321
 11 |    14641 :    1331 |    2835 :     2838 |     3476 |  0:00:00 | 3.315
 13 |    28561 :    2197 |    5532 :     5538 |     9008 |  0:00:01 | 3.36
 17 |    83521 :    4913 |   13429 :    13432 |    22437 |  0:00:05 | 3.355
 19 |   130321 :    6859 |   20779 :    20782 |    43216 |  0:00:08 | 3.376

total time: 0:00:16.903033


In [8]:
# goal: pickle d for later use
# WARNING: will overwrite existing file

with open("data.pkl", "wb") as f:
    pickle.dump(d, f)

In [9]:
# calculate factorization patterns of f where L(C,T) = 1 + a*T^g + T^{2g} mod 2
def good_patterns(g):
    F = GF(2)
    R.<x> = F[]
    x_to_i = [ R(1) ]
    while len(x_to_i) < 2*g+3:
        x_to_i.append(x_to_i[-1]*x)

    # enumerate partitions of 2g+2
    for p in Partitions(2*g+2):
        p = list(p)
        
        L = prod([ x_to_i[mi]-1 for mi in p ],R(1))
        assert L % (x-1)^2 == 0
        L //= (x-1)^2
        assert L.degree() == 2*g
        
        c = L.coefficients(sparse=False)
        if all([ ci == 0 for ci in c[1:g] ]):
            yield p

            

In [10]:
list(good_patterns(2))

[[6],
 [4, 2],
 [4, 1, 1],
 [3, 3],
 [2, 2, 2],
 [2, 2, 1, 1],
 [2, 1, 1, 1, 1],
 [1, 1, 1, 1, 1, 1]]

In [11]:
list(good_patterns(3))

[[6, 2], [6, 1, 1], [3, 3, 2], [3, 3, 1, 1]]

In [12]:
list(good_patterns(4))

[[8, 2],
 [8, 1, 1],
 [4, 4, 2],
 [4, 4, 1, 1],
 [4, 2, 2, 2],
 [4, 2, 2, 1, 1],
 [4, 2, 1, 1, 1, 1],
 [4, 1, 1, 1, 1, 1, 1],
 [2, 2, 2, 2, 2],
 [2, 2, 2, 2, 1, 1],
 [2, 2, 2, 1, 1, 1, 1],
 [2, 2, 1, 1, 1, 1, 1, 1],
 [2, 1, 1, 1, 1, 1, 1, 1, 1],
 [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]

In [13]:
list(good_patterns(5))

[[10, 2], [10, 1, 1], [5, 5, 2], [5, 5, 1, 1]]